[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_diffsynth_practive/blob/main/anima_minimal_colab.ipynb)

# Minimal Anima T2I — extracted DiffSynth path

ComfyUI / Diffusers / DiffSynth를 설치하거나 별도 git clone하지 않습니다. 이 저장소의 최소 모델 정의와 일반 Python 라이브러리만 사용합니다. 모델 가중치는 공개 Hugging Face `circlestone-labs/Anima`에서 인증 없이 받습니다.


In [ ]:
# 1. 일반 Python 라이브러리만 설치
!pip -q install -U 'transformers>=4.57,<4.58' 'huggingface_hub>=0.36' safetensors einops sentencepiece pillow tqdm


In [ ]:
# 2. 이 실습 저장소만 clone
from pathlib import Path
import subprocess, sys, torch

REPO = 'https://github.com/HisameOgasahara/tmp_diffsynth_practive.git'
ROOT = Path('/content/tmp_diffsynth_practive')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT / 'src'))
print(torch.__version__, torch.cuda.get_device_name() if torch.cuda.is_available() else 'NO CUDA')


In [ ]:
# 3. Baseline 생성값 — DiffFlowDiT_test/config/generation.json에서 옮김
PROMPT = '@ogipote, 1girl, plana (blue archive), upper body, \\n, solo, cute, cheerful, innocent, expressionless, looking_at_viewer, alternte costume, white school uniform, blue_ribbon, white_socks, school_theme, sitting, holding_tablet, sunlight, soft_lighting, gentle_breeze, floating_particles, futuristic_city, academy_city, glass_building, blue_and_white_theme, pastel_colors, vibrant_colors, depth_of_field, wallpaper' # @param {type:'string'}
NEGATIVE_PROMPT = 'worst quality, low quality, score_1, score_2, score_3, blurry, jpeg artifacts' # @param {type:'string'}
SEED = 1113280783077040 # @param {type:'integer'}
STEPS = 30 # @param {type:'integer'}
CFG = 4.0 # @param {type:'number'}
WIDTH = 1216 # @param {type:'integer'}
HEIGHT = 832 # @param {type:'integer'}
DENOISE = 1.0 # @param {type:'number'}
SHIFT = 3.0 # @param {type:'number'}

DEVICE = 'cuda'
DTYPE = torch.float16


In [ ]:
# 4. 공개 HF에서 Anima 가중치 다운로드 — token/auth 없음
from anima_core.runtime import download_weights
weights = download_weights()
weights


In [ ]:
# 5. Text Encoder -> positive / negative embedding
import gc
from anima_core.runtime import load_text_encoder, load_tokenizers, encode_prompt

qwen_tokenizer, t5_tokenizer = load_tokenizers()
text_encoder = load_text_encoder(weights['text_encoder'], DEVICE, DTYPE)
positive_raw, positive_t5 = encode_prompt(text_encoder, qwen_tokenizer, t5_tokenizer, PROMPT, DEVICE, DTYPE)
negative_raw, negative_t5 = encode_prompt(text_encoder, qwen_tokenizer, t5_tokenizer, NEGATIVE_PROMPT, DEVICE, DTYPE)

del text_encoder
gc.collect(); torch.cuda.empty_cache()
print(positive_raw.shape, positive_t5.shape)


In [ ]:
# 6. Anima DiT + text adapter -> CFG Euler sampling
from anima_core.runtime import load_dit, adapt_conditioning, sample_euler

dit = load_dit(weights['dit'], DEVICE, DTYPE)
positive = adapt_conditioning(dit, positive_raw, positive_t5)
negative = adapt_conditioning(dit, negative_raw, negative_t5)
del positive_raw, negative_raw, positive_t5, negative_t5
gc.collect(); torch.cuda.empty_cache()

latents = sample_euler(
    dit, positive, negative, HEIGHT, WIDTH, SEED, STEPS, CFG,
    denoise=DENOISE, shift=SHIFT, device=DEVICE, dtype=DTYPE,
)
del dit, positive, negative
gc.collect(); torch.cuda.empty_cache()
print(latents.shape)


In [ ]:
# 7. VAE decode -> PNG
from IPython.display import display
from anima_core.runtime import load_vae, decode_image

vae = load_vae(weights['vae'], DEVICE, DTYPE)
image = decode_image(vae, latents, DEVICE)
OUTPUT = Path('/content/anima_minimal.png')
image.save(OUTPUT)
display(image)
print(OUTPUT)
